In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import os, json
import numpy as np
from PIL import Image
import shutil

In [ ]:

def augment_and_save(src_dir, dst_dir, multiplier=3):
    aug = transforms.Compose([
        transforms.RandomRotation(45),
        transforms.ColorJitter(brightness=0.3, saturation=0.4, hue=0.15),
        transforms.RandomHorizontalFlip(p=1.0),
    ])
    for class_name in os.listdir(src_dir):
        src_class = os.path.join(src_dir, class_name)
        dst_class = os.path.join(dst_dir, class_name)
        os.makedirs(dst_class, exist_ok=True)
        for fname in os.listdir(src_class):
            fpath = os.path.join(src_class, fname)
            if not os.path.isfile(fpath): continue
            img = Image.open(fpath).convert("RGB")
            shutil.copy(fpath, dst_class)
            for i in range(multiplier):
                aug(img).save(os.path.join(dst_class, f"aug_{i}_{fname}"))

if not os.path.exists("dataset_augmented"):
    augment_and_save("dataset_recognition", "dataset_augmented", multiplier=3)

DATA_DIR = "dataset_augmented"
OOD_DIR = "dataset_ood"


In [ ]:
# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Menggunakan device: {device}")

NUM_CLASSES = 4 # Pisang, Tomat, Alpukat, Pir
EPOCHS = 20
BATCH_SIZE = 16

In [ ]:
# 2. Data Loading and Transform
transform_train = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.5, hue=0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Load Known Food
full_train = datasets.ImageFolder(root=DATA_DIR, transform=transform_train)
full_val   = datasets.ImageFolder(root=DATA_DIR, transform=transform_val)

targets    = [s[1] for s in full_train.samples]
train_idx, val_idx = train_test_split(
    list(range(len(full_train))), test_size=0.2, stratify=targets, random_state=42
)

train_loader = DataLoader(Subset(full_train, train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(Subset(full_val,   val_idx),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Load OOD Data
try:
    ood_dataset = datasets.ImageFolder(root=OOD_DIR, transform=transform_train)
    ood_loader = DataLoader(ood_dataset, batch_size=BATCH_SIZE, shuffle=True)
    ood_iter   = iter(ood_loader)
    print(f"✅ OOD Dataset dimuat dengan {len(ood_dataset)} gambar.")
except Exception as e:
    print(f"🚨 ERROR memuat OOD Dataset: {e}")
    print("Pastikan ada folder 'dataset_ood' yang berisi subfolder (misal 'non_food') di Colab-mu!")
    raise e

In [ ]:
# OSR Architecture with EfficientB3
class OpenSetClassifier(nn.Module):
    def __init__(self, n_classes, energy_threshold=-10.0):
        super().__init__()
        self.backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

        # Buka features.4 dan features.5 juga — lebih banyak yang bisa adapt
        for name, p in self.backbone.named_parameters():
            if not any(name.startswith(s) for s in
                       ['features.4', 'features.5', 'features.6', 'features.7', 'classifier']):
                p.requires_grad = False

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1536, n_classes)
        )
        self.energy_threshold = energy_threshold

    def forward(self, x):
        feat_map = self.backbone.features[:6](x)
        out      = self.backbone.features[6:](feat_map)
        out      = self.backbone.avgpool(out)
        logits   = self.backbone.classifier(out.flatten(1))
        energy   = -torch.logsumexp(logits, dim=-1)
        probs    = F.softmax(logits, dim=-1)
        conf, pred = probs.max(dim=-1)
        return {
            "logits": logits, "pred_class": pred,
            "confidence": conf, "energy": energy,
            "is_known": (energy < self.energy_threshold),
            "feature_map": feat_map
        }

model = OpenSetClassifier(n_classes=NUM_CLASSES).to(device)

In [ ]:
optimizer = optim.AdamW([
    {"params": model.backbone.features[4:6].parameters(), "lr": 5e-6},
    {"params": model.backbone.features[6:].parameters(),  "lr": 1e-5},
    {"params": model.backbone.classifier.parameters(),    "lr": 2e-4},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

criterion = nn.CrossEntropyLoss()

# Training
os.makedirs("weights", exist_ok=True)
best_acc = 0.0

MARGIN = 25.0
LAMBDA = 0.5

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)
        try:
            ood_images, _ = next(ood_iter)
        except StopIteration:
            ood_iter = iter(ood_loader)
            ood_images, _ = next(ood_iter)
        ood_images = ood_images.to(device)

        optimizer.zero_grad()

        # Loss 1: Classification (Known Food)
        out_known = model(images)
        loss_ce   = criterion(out_known["logits"], labels)

        # Loss 2: OE Loss (OOD)
        out_ood    = model(ood_images)
        energy_ood = -torch.logsumexp(out_ood["logits"], dim=-1)
        loss_oe    = torch.mean(torch.relu(-MARGIN - energy_ood))

        # Total loss
        loss = loss_ce + LAMBDA * loss_oe

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()

    scheduler.step()

    # Validate
    model.eval()
    correct, total = 0, 0
    all_energies_known = []
    all_energies_ood   = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            correct += (out["pred_class"] == labels).sum().item()
            total   += labels.size(0)
            all_energies_known.extend(out["energy"].cpu().tolist())

        for ood_images, _ in ood_loader:
            out_ood = model(ood_images.to(device))
            all_energies_ood.extend(out_ood["energy"].cpu().tolist())

    val_acc     = 100 * correct / total
    mean_known  = np.mean(all_energies_known)
    mean_ood    = np.mean(all_energies_ood)
    gap         = mean_ood - mean_known

    print(f"Epoch {epoch+1:2d}: acc={val_acc:.1f}% | E_known={mean_known:.2f} | E_ood={mean_ood:.2f} | gap={gap:.2f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "weights/openset.pt")
        print(f"   ✅ Saved! acc={best_acc:.1f}%")

Epoch 1/25:  55%|█████▍    | 68/124 [14:44<12:08, 13.01s/it]


KeyboardInterrupt: 

In [ ]:
model.eval()
energies_known = []
energies_ood   = []

with torch.no_grad():
    for images, _ in val_loader:
        out = model(images.to(device))
        energies_known.extend(out["energy"].cpu().tolist())

    for images, _ in ood_loader:
        out = model(images.to(device))
        energies_ood.extend(out["energy"].cpu().tolist())

ek = np.array(energies_known)
eo = np.array(energies_ood)

p95_known = np.percentile(ek, 95)
p5_ood    = np.percentile(eo, 5)
threshold = (p95_known + p5_ood) / 2.0

y_true  = [0]*len(ek) + [1]*len(eo)
y_score = list(ek) + list(eo)
auroc   = roc_auc_score(y_true, y_score)

with open("weights/osr_config.json", "w") as f:
    json.dump({
        "energy_threshold": threshold,
        "n_classes": NUM_CLASSES,
        "class_names": full_train.classes
    }, f, indent=2)